In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from matplotlib import pyplot as plt
import torchvision.transforms as T
import os
from PIL import Image
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

# 配置信息
print("=" * 60)
print("PyTorch 版本信息:")
print(f"  PyTorch 版本: {torch.__version__}")
print(f"  CUDA 版本: {torch.version.cuda}")
print(f"  cuDNN 版本: {torch.backends.cudnn.version()}")

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print("\n" + "=" * 60)
    print("GPU 配置信息:")
    print(f"  检测到 GPU 数量: {torch.cuda.device_count()}")
    print(f"  当前使用 GPU: {torch.cuda.current_device()}")
    print(f"  GPU 名称: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0)
    total_mem = gpu_mem.total_memory / 1024**3
    print(f"  GPU 显存总量: {total_mem:.2f} GB")
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    print(f"  已分配显存: {allocated:.2f} GB")
    print(f"  已缓存显存: {reserved:.2f} GB")
    torch.backends.cudnn.benchmark = True
    print(f"\n  cuDNN benchmark: 启用")
    print("\n" + "=" * 60)
    print("训练设备: GPU (cuda:0)")
    print("=" * 60)
else:
    device = torch.device("cpu")
    print("\n" + "=" * 60)
    print("GPU 不可用，将使用 CPU 进行训练")
    print("=" * 60)


In [ ]:
def plot_curve(data):#画出损失值的下降曲线
    fig = plt.figure()
    plt.plot(range(len(data)), data, color='blue')
    plt.legend(['value'], loc='upper right')
    plt.xlabel('step')
    plt.ylabel('value')
    plt.savefig('two-loss.png')
    plt.show()

In [ ]:
def plot_image(img, label, name):
    # 确定要显示的图片数量，取批次大小和4的较小值
    num_images = min(img.shape[0], 4)
    plt.figure(figsize=(10, 8))
    
    for i in range(num_images):
        plt.subplot(2, 2, i + 1)
        plt.tight_layout()
        plt.imshow(img[i].permute(1, 2, 0))  # 使用permute代替transpose，效果相同
        plt.title("{}: {}".format(name, label[i].item()))
        plt.xticks([])
        plt.yticks([])
    
    plt.show()

In [ ]:
def one_hot(label, depth=2):
    # 创建一个全零张量，形状为 (batch_size, depth)，设备与 label 一致
    out = torch.zeros(label.size(0), depth, device=label.device)
    # 将 label 转换为长整型张量，并保持在同一设备上
    idx = torch.tensor(label, dtype=torch.long, device=label.device).view(-1, 1)
    # 使用 scatter_ 方法将对应位置的值设置为 1
    out.scatter_(dim=1, index=idx, value=1)
    return out

In [ ]:
# 随机种子
RANDOM_SEED = 123

class MyDataset(Dataset):
    """
    自定义数据集类
    
    参数说明：
    - flag: 数据集类型，'train' 或 'test'
    - seed: 随机种子，用于控制数据划分的随机性
    
    使用相同的seed可以确保每次运行得到相同的数据划分，便于实验复现
    """
    
    def __init__(self, flag='train', seed=None):
        # 设置随机种子
        if seed is not None:
            import random
            random.seed(seed)
            import numpy as np
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
        
        assert flag in ['train', 'test', 'valid']
        self.flag = flag
        
        # 获取所有图片路径
        all_images = []
        for i in os.listdir("../data/binary/0"):
            all_images.append((os.path.join("../data/binary/0/", i), 0))
        for i in os.listdir("../data/binary/1"):
            all_images.append((os.path.join("../data/binary/1/", i), 1))
        
        # 划分训练集和测试集 (80% 训练, 20% 测试)
        import random
        random.shuffle(all_images)
        split_idx = int(len(all_images) * 0.8)
        
        self.Animedata_path = all_images[:split_idx]
        self.Animedata_testpath = all_images[split_idx:]
        
        self.resize_transform = T.Resize((96, 96))
        self.transforms = T.ToTensor()
            
    def __getitem__(self, index):
        if self.flag == 'train':
            image_path, label = self.Animedata_path[index]
            image = Image.open(image_path)
            image = self.resize_transform(image)
            image = self.transforms(image).float()
            return image, torch.tensor(label)
        elif self.flag == 'test':
            image_path, label = self.Animedata_testpath[index]
            image = Image.open(image_path)
            image = self.resize_transform(image)
            image = self.transforms(image).float()
            return image, torch.tensor(label)
    
    def __len__(self):
        if self.flag == 'train':
            return len(self.Animedata_path)
        elif self.flag == 'test':
            return len(self.Animedata_testpath)

In [ ]:
batch_size=64 #每一个训练的Step 送往模型中的图片的数量 当你数据很多的时候 可以设得很大 当样本分布分散时 可以设的比较大 当图片比较少 或者样本分布 比较集中 可以设一个比较小的值

train_loader = torch.utils.data.DataLoader(MyDataset(flag='train'),batch_size=batch_size, shuffle=True)
#迭代器 在for循环中 按循环次数 调用getitem函数 返回数据和标签 shuffle 对样本进行随机排序 返回训练集的数据
test_loader= torch.utils.data.DataLoader(MyDataset(flag='test'),batch_size=batch_size, shuffle=False)
#迭代器 返回测试集的数据

x,y=next(iter(train_loader)) # iter iteration 迭代 next函数之后 可以返回一组getitem中的数据

# x是图片 y是标签
plot_image(x,y,"AnimeFaces")#画出 读取的 图片
y

In [ ]:
class Net1(nn.Module):#Net类 功能是 描述清楚 模型中包含的网络层数 每一层的类型 数据在模型中的一个传播路径
#resnet18 densenet senet googlenet
    def __init__(self):#构造函数 声明 所有需要在模型中出现的层
        super(Net1, self).__init__()
        self.conv1=nn.Conv2d(in_channels=4,out_channels=6,kernel_size=5)#96 96 4 ->92 92 6
        self.pool1=nn.MaxPool2d(kernel_size=2)#46 46 6
        self.conv2=nn.Conv2d(in_channels=6,out_channels=16,kernel_size=5)#42 42 16
        self.pool2=nn.MaxPool2d(kernel_size=2)#21 21 16
        self.fc1 = nn.Linear(21*21*16, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x=self.pool1(x)
        x = F.relu(self.conv2(x))
        x=self.pool2(x)
        x = x.view(x.size(0), 21*21*16)
        x = F.relu(self.fc1(x)) #激活函数relu可以替换leak—relu
        x = F.relu(self.fc2(x))

        x = self.fc3(x)

        return x#返回模型的预测结果

In [ ]:
net = Net1()#建立Net类的实例net net是我们构建的模型
net = net.to(device) # 把模型搬到GPU上
#optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
# optimizer = torch.optim.Adam(net.parameters(), lr=0.001, betas=(0.9,0.999), eps=1e-8, weight_decay=0, amsgrad=False)
optimizer = torch.optim.Adam(net.parameters(), lr=0.0001, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-5, amsgrad=False)
#rmprop adagrad adam sgdm 学习率衰减和动量
#优化器 功能：实现梯度下降 net.parameters()模型的所有参数 
#lr learn rate 学习率 功能是 控制训练的速度 设太大会导致 损失值波动 太小则损失值下降速度慢
#momentum 动量 动量越大 越可能快速通过 平台期 鞍点和局部最优点 太大会导致损失值波动

In [ ]:
# 导入tqdm用于显示训练进度条
from tqdm import tqdm

# 设置随机种子
RANDOM_SEED = 123
import random
random.seed(RANDOM_SEED)
import numpy as np
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

train_loss = []

# ======================= 内存优化开始 =======================
import gc

# 设置训练轮次
num_epochs = 50  # 控制训练的轮次，epoch数量越大则训练的次数越多

# 用于存储每个epoch的平均损失
epoch_losses = []

for epoch in range(num_epochs):
    # ======================= 周期开始时的内存清理 =======================
    gc.collect()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # 初始化本epoch的损失累计
    epoch_loss = 0.0
    num_batches = 0
    

    for batch_idx, (x, y) in enumerate(tqdm(train_loader, 
                                               desc=f"Epoch {epoch+1}/{num_epochs}", 
                                               ncols=80,
                                               leave=True)):
        # 调用GPU训练
        x = x.to(device)
        y = y.to(device)

        out = net(x)

        y_onehot = one_hot(y)

        loss = F.mse_loss(out, y_onehot)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 累计本epoch的损失
        epoch_loss += loss.item()
        num_batches += 1

        train_loss.append(loss.item())
        
        del out, y_onehot, loss
    
    # 计算本epoch的平均损失
    avg_loss = epoch_loss / num_batches if num_batches > 0 else 0
    epoch_losses.append(avg_loss)
    
    # 打印本epoch的结果
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.6f}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n训练完成!")

plot_curve(train_loss)



In [ ]:
# ======================= 测试阶段的内存优化 =======================
# 使用torch.no_grad()上下文管理器禁用梯度计算
# 这是PyTorch中最重要的内存优化技巧之一：
# 1. 禁用梯度跟踪和梯度存储，大幅减少显存占用
# 2. 可以减少约30-40%的显存使用
# 3. 推理/验证阶段必须使用，训练阶段在特定场景下也可使用

total_correct = 0#用来记录 所有数据中预测正确的样本数量
with torch.no_grad():  # 重要：禁用梯度计算，显著减少显存占用
    for x,y in test_loader:#从测试数据中 按batch依次读出图片和标签
        # 使用GPU
        x = x.to(device)
        y = y.to(device)
        #x  = x.view(x.size(0), 96*96*4)#拉伸成向量
        out = net(x)#把图片x输入模型

        pred = out.argmax(dim=1)#获取输出结果 最大值所在的维度 即为预测的类别信息
        #pred是预测的类别
        correct = pred.eq(y).sum().float().item()#比较数值标签y和预测的数值类别pred 并统计两者相同的次数
        #correct为y与pred相同的样本的数量
        total_correct += correct#把本batch中预测正确的样本数量 记录的 总预测正确的样本数量中

        # 在每个batch结束后清理中间变量
        del out, pred, x, y

# 验证完成后进行内存清理
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

#准确率 为 预测正确的样本数量/所有的样本数量
total_num = len(test_loader.dataset)#计算测试集中所有的样本数量
acc = total_correct / total_num#计算准确率 预测正确的样本数量/所有的样本数量
print('test acc:', acc)#打印准确率

# ======================= 可视化阶段的内存优化 =======================
# 将数据移动到CPU上进行可视化，避免占用GPU显存

x, y = next(iter(test_loader))#从测试集中读几张图
# 将数据移动到正确的设备上
x = x.to(device)
y = y.to(device)

# 同样使用no_grad进行推理
with torch.no_grad():
    out = net(x)#拉伸成一维向量
    pred = out.argmax(dim=1)#获取预测的类别信息

# 将 x 和 pred 移动到 CPU 上，以便绘制图像
# 注意：在绘图完成后及时删除GPU上的张量
x = x.cpu()
pred = pred.cpu()
y = y.cpu()

# 清理GPU上的张量
del out, x, y
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

plot_image(x, pred, 'test')#把随机挑选的测试集图片和预测结果画出来

In [ ]:
torch.save(net.state_dict(), 'two_model.pth')

In [ ]:
import cv2
from PIL import Image
import numpy as np
# 灰度图读入
img = cv2.imread('10.png', 1)
#ret,dst=cv2.threshold(img,150,255,cv2.THRESH_BINARY)

img_r1 = cv2.selectROIs("roi",img,False,False)

#img_r = cv2.selectROIs("roi",img,False,False)

img1=img[img_r1[0][1]:img_r1[0][1]+img_r1[0][3],img_r1[0][0]:img_r1[0][0]+img_r1[0][2]]

# 加载模型
loaded_model = Net1()
loaded_model.load_state_dict(torch.load('two_model.pth'))
# loaded_model = loaded_model.to(device)


resize_transform = T.Resize((96, 96))
transforms=T.ToTensor()

image=img[img_r1[0][1]:img_r1[0][1]+img_r1[0][3],img_r1[0][0]:img_r1[0][0]+img_r1[0][2]]
cv2.waitKey(0)
cv2.destroyAllWindows()


image = Image.fromarray(image)

image=resize_transform(image)
image=transforms(image).float()

out = loaded_model(x)

pred = out.argmax(dim=1)
print(out)
print(pred)